# Professional Wake Word Engine — Two-Layer Positive-Only Enrollment (v9)

Conservative Layer-2 statistics for 3–5 real enrollment clips, with clip-balanced enrollment, public within-class PCA, deterministic calibration sampling, and safer streaming evaluation.

**v9 fixes the bug that made detection never fire, adds the missing interactive detection demo with plots, and wires in TensorBoard for training + evaluation tracking.** v8 added the production/verification layer: score-distribution and DET-curve visualization, an actual PyTorch↔ONNX numerical parity check (export bugs are otherwise invisible), real INT8 dynamic quantization with a measured size/parity trade-off, a standalone `onnxruntime`-only deployment inference path that doesn't depend on the training notebook at all, an environment/version snapshot for reproducibility, and lightweight sanity tests for the trigger state machine.


## 1. Setup

In [ ]:
!pip install -q torch torchaudio numpy scipy scikit-learn onnx onnxruntime matplotlib tensorboard


In [ ]:
import os, glob, random, math, json
from collections import defaultdict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

### Reproducibility note

The install cell above deliberately does **not** pin versions, so the notebook keeps
working as PyTorch/torchaudio evolve. That's a trade-off: it also means the exact
behavior can drift between runs months apart (e.g. `torchaudio.datasets.SPEECHCOMMANDS`
or ONNX opset support changing). The cell below prints the resolved versions so you can
pin them retroactively (`pip freeze > requirements.txt`) once you have a profile you're
happy with, and reproduce that exact environment later.

In [ ]:
import platform, sys, subprocess

def _pkg_version(name):
    try:
        return __import__(name).__version__
    except Exception:
        return "n/a"

print("=== Environment snapshot (for reproducibility / bug reports) ===")
print(f"python:      {sys.version.split()[0]}")
print(f"platform:    {platform.platform()}")
print(f"torch:       {_pkg_version('torch')}")
print(f"torchaudio:  {_pkg_version('torchaudio')}")
print(f"numpy:       {_pkg_version('numpy')}")
print(f"scipy:       {_pkg_version('scipy')}")
print(f"onnx:        {_pkg_version('onnx')}")
print(f"onnxruntime: {_pkg_version('onnxruntime')}")
print(f"sklearn:     {_pkg_version('sklearn')}")
if torch.cuda.is_available():
    print(f"cuda:        {torch.version.cuda}, device={torch.cuda.get_device_name(0)}")
else:
    print("cuda:        not available (running on CPU)")

# NOTE: the pip install cell above is intentionally unpinned so the notebook keeps
# working with future library releases. If you hit a reproducibility issue months
# later, pin the exact versions printed here (e.g. `torch==X.Y.Z`) and re-run.


## TensorBoard — training & evaluation tracking

Logs Layer-1 training curves (loss/accuracy, train vs val) and Layer-2
evaluation metrics (calibration threshold, FAR/FRR/EER, score-distribution
and DET-curve figures, and the detection-demo plot in Section 12.5) to
TensorBoard -- one dashboard covers both training and testing, updating live
as later cells run. Each run gets its own timestamped log directory under
`./runs` so re-running the notebook doesn't overwrite a previous run's curves.

In [ ]:
import datetime
from torch.utils.tensorboard import SummaryWriter

RUN_NAME = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_LOGDIR = f"./runs/{RUN_NAME}"
writer = SummaryWriter(log_dir=RUN_LOGDIR)
print(f"TensorBoard logging to {RUN_LOGDIR}")

%load_ext tensorboard
%tensorboard --logdir ./runs

## 2. Config

In [ ]:
SR         = 16000
CLIP_SEC   = 1.0
CLIP_LEN   = int(SR * CLIP_SEC)
N_MELS     = 40
EMBED_DIM  = 128          # Layer-1 and Layer-2 both operate on this embedding size

BACKBONE      = "dscnn"   # switch to "repcnn" once you've implemented it below
TEMPORAL_HEAD = "gru"     # "gru" (default: preserves phoneme ordering, heavier)
                           # "pool" (plain global-average-pool: lighter/faster,
                           # use this if targeting the tightest embedded budgets)

DATA_ROOT       = "./data"                       # Speech Commands auto-downloads here
UNIVERSAL_CKPT  = "./universal_encoder.pt"        # Layer-1 best-model checkpoint, reuse forever
RESUME_CKPT     = "./universal_encoder.resume.pt" # Layer-1 latest-epoch checkpoint, for resuming
RESUME_TRAINING = False                           # set True to continue from RESUME_CKPT if present
ENROLLMENT_DIR  = "./my_wakeword_enrollment"      # put your 3-5 positive clips here
POSITIVE_EVAL_DIR = "./my_wakeword_eval_positive" # OPTIONAL: 20-100 additional wake-word
                                                    # recordings, held out from enrollment,
                                                    # used ONLY in Section 10 for a real FRR
                                                    # estimate. Leave the folder empty/missing
                                                    # and Section 10 falls back to the weaker
                                                    # leave-one-clip-out estimate, with a warning.
PROFILE_PATH    = "./wakeword_profile.json"       # Layer-2 output

# FIXED (v7): these folders were never created automatically, so a missing
# ENROLLMENT_DIR silently produced an empty glob() and a confusing assert
os.makedirs(ENROLLMENT_DIR, exist_ok=True)
os.makedirs(POSITIVE_EVAL_DIR, exist_ok=True)

# Episodic training config for Layer 1 (many-keyword metric learning)
N_WAY               = 10     # keyword classes per episode
N_SUPPORT           = 5
N_QUERY             = 5
EPISODES_PER_EPOCH  = 100
VAL_EPISODES        = 30
MAX_EPOCHS           = 60
PATIENCE             = 10
LR                   = 1e-3

# --- Background-noise train/val/test split (Section 5) --------------------
# torchaudio's SPEECHCOMMANDS "_background_noise_" folder is NEVER assigned
# to the validation or testing split by the dataset itself -- it only ever
# lands in "training". Left alone, that means validation episodes and the
# Section 10 held-out test both silently reuse (or in the test case, ENTIRELY
# SKIP) noise-impostor evaluation. Since there are only a handful of long
# noise recordings (~6 files) to work with, we can't hold whole files out
# without losing most noise diversity from one split or another -- instead
# each file is cut along TIME into three non-overlapping contiguous chunks
# per the fractions below. Coarser than genuinely separate recordings, but
# the splits are now non-overlapping audio, not literally identical samples.
NOISE_SPLIT_FRACTIONS = {"train": (0.00, 0.65), "val": (0.65, 0.75), "calib": (0.75, 0.85), "test": (0.85, 1.00)}

# --- Layer 2 enrollment config ---------------------------------------------
N_ENROLL_AUGS = 4      # mild augmented variants generated PER raw enrollment clip,
                        # to give the covariance estimate more support without
                        # needing more real recordings from you

# Augmented variants of the SAME raw clip are correlated with each other, not
# independent draws -- 5 raw clips x 5 variants each is still much closer to
# 5 independent pieces of evidence than to 25. A full 128x128 covariance
# estimated from that little independent information is fragile regardless
# of shrinkage. COV_MODE controls how much of that risk you're taking on:
#   "full"     -- full 128x128 covariance (higher capacity, most fragile
#                 with only ~5 independent recordings; use once you've
#                 enrolled with many more varied real clips)
#   "diagonal" -- 128 independent variances only (no cross-dimension
#                 correlation modeled; far fewer parameters to estimate per
#                 independent sample, the safest default for 3-5 clips)
#   "pca_diag" -- project into a low-dimensional subspace learned from public
#                 within-class speech variation, then fit only diagonal variance.
#                 This is the recommended mode for 3–5 real enrollment clips.
#   "pca"      -- legacy full covariance in PCA space; experimental only.
#                 PUBLIC DATA (never from your enrollment clips -- that would
#                 be circular), then fit a full covariance inside that
#                 smaller space. This mirrors how PLDA-style speaker
#                 verification handles limited-enrollment covariance: learn
#                 the subspace from abundant background data, then only fit
#                 statistics for the new (wake word / speaker) inside it.
COV_MODE = "pca_diag"
PCA_DIM  = 8
COV_SHRINKAGE        = 0.75
MIN_VARIANCE         = 1e-3           # numerical / anti-overconfidence floor
                         # total enrollment embedding count (raw clips * (1+N_ENROLL_AUGS))

# --- Detection calibration -- centralized here, not buried in a later cell,
# since it's the one number you're most likely to want to tune. ------------
TARGET_FAR = 1e-4       # target false-accept rate for CONTINUOUS listening,
                        # not a one-shot classification rate. NOTE: with this
                        # few calibration samples, the achieved threshold is a
                        # Gamma-tail EXTRAPOLATION targeting this rate, not a
                        # directly measured one -- Section 10's held-out test
                        # is what tells you the actual achieved FAR/FRR.

# --- Streaming / temporal confirmation (Sections 11-12) --------------------
SCORING_HOP_SEC          = 0.1   # fine-grained scoring hop; controls responsiveness/latency
MIN_CONFIRM_SPACING_SEC  = 0.3   # minimum REAL TIME between counted confirmations -- enforced
                                   # against the stream clock, not a fixed-stride counter, so a
                                   # run of VAD misses can't distort the actual spacing (see Sec 11)
CONFIRM_WINDOWS          = 3
HANGOVER_SEC             = 1.5

## 3. Feature extraction — LFBE + PCEN

PCEN (Per-Channel Energy Normalization) replaces plain log-compression. It's
what makes the model robust to loudness swings and far-field/near-field
differences — this is the same technique used in Google's production KWS
research. Implemented manually here since torchaudio doesn't ship it built in.

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=400, hop_length=160, n_mels=N_MELS, power=2.0
)

def pcen(mel_power, alpha=0.98, delta=2.0, r=0.5, s=0.025, eps=1e-6):
    # mel_power: (..., n_mels, time) — a power spectrogram (non-log)
    frames = mel_power.unbind(-1)
    m = frames[0]
    M = [m]
    for f in frames[1:]:
        m = (1 - s) * m + s * f
        M.append(m)
    M = torch.stack(M, dim=-1)
    out = (mel_power / (eps + M).pow(alpha) + delta).pow(r) - delta ** r
    return out

def load_wav(path, sr=SR):
    wav, in_sr = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if in_sr != sr:
        wav = torchaudio.functional.resample(wav, in_sr, sr)
    return wav.squeeze(0)

def fix_length_center(wav, length=CLIP_LEN):
    # Center crop/pad only -- NEVER random crop for positive/enrollment audio.
    # Random cropping can slice off part of the wake word (a real bug from an
    # earlier version of this pipeline) -- center alignment keeps the full
    # word intact.
    n = wav.shape[0]
    if n == length:
        return wav
    if n > length:
        start = (n - length) // 2
        return wav[start:start + length]
    pad = length - n
    left = pad // 2
    return F.pad(wav, (left, pad - left))

def fix_length_random(wav, length=CLIP_LEN):
    # Random crop is fine for Layer-1 training (many keyword classes, lots of
    # data) -- just never use it for your own positive enrollment clips.
    n = wav.shape[0]
    if n == length:
        return wav
    if n > length:
        start = random.randint(0, n - length)
        return wav[start:start + length]
    pad = length - n
    left = random.randint(0, pad)
    return F.pad(wav, (left, pad - left))

def time_shift(wav, max_shift=0.15):
    # Zero-padded shift, NOT torch.roll -- roll wraps audio around and
    # produces physically unrealistic "shifted" clips.
    n = wav.shape[0]
    shift = int(random.uniform(-max_shift, max_shift) * n)
    out = torch.zeros_like(wav)
    if shift > 0:
        out[shift:] = wav[:n - shift]
    elif shift < 0:
        out[:n + shift] = wav[-shift:]
    else:
        out = wav
    return out

def gain_jitter(wav, db_range=(-6, 6)):
    db = random.uniform(*db_range)
    return wav * (10 ** (db / 20))

def mix_noise(wav, noise_wav, snr_db_range=(0, 20)):
    noise_seg = fix_length_random(noise_wav, wav.shape[0])
    sig_power = wav.pow(2).mean().clamp_min(1e-8)
    noise_power = noise_seg.pow(2).mean().clamp_min(1e-8)
    snr_db = random.uniform(*snr_db_range)
    target_noise_power = sig_power / (10 ** (snr_db / 10))
    scaled_noise = noise_seg * torch.sqrt(target_noise_power / noise_power)
    return wav + scaled_noise

def wav_to_features(wav):
    mel = mel_transform(wav.unsqueeze(0))       # (1, n_mels, time), power
    feat = pcen(mel)
    feat = (feat - feat.mean()) / (feat.std() + 1e-6)
    return feat.unsqueeze(0)                     # (1, 1, n_mels, time)

## 4. Backbone — swappable encoder

`build_encoder("dscnn")` today. To move to RepCNN later: implement
`RepCNNEncoder` with the exact same `forward(x) -> (batch, EMBED_DIM)`
L2-normalized-output contract, register it in `build_encoder`, and change
`BACKBONE` in the config cell. Nothing else in this notebook needs to change.

`TEMPORAL_HEAD` is a real, wired-in choice, not just a comment: `"gru"`
keeps sequence/phoneme-ordering information at the cost of extra parameters
and compute; `"pool"` collapses straight to global-average-pooling like a
plain DS-CNN classifier, trading that sequence information for a smaller,
faster model. The parameter breakdown printed below shows exactly what the
temporal head costs relative to the rest of the network, so the trade-off
is visible rather than assumed.

In [ ]:
class DSConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride,
                                    padding=1, groups=in_ch, bias=False)
        self.dw_bn = nn.BatchNorm2d(in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pw_bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.dw_bn(self.depthwise(x)))
        x = self.relu(self.pw_bn(self.pointwise(x)))
        return x

class DSCNNEncoder(nn.Module):
    '''Backbone contract: forward(x) where x is (batch, 1, n_mels, time)
    -> returns (batch, EMBED_DIM), L2-normalized.'''
    def __init__(self, n_mels=N_MELS, embed_dim=EMBED_DIM, temporal_head=TEMPORAL_HEAD):
        super().__init__()
        if temporal_head not in ("gru", "pool"):
            raise ValueError(f"Unknown temporal_head: {temporal_head}")
        self.temporal_head = temporal_head
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.ds_blocks = nn.Sequential(
            DSConvBlock(32, 64), nn.MaxPool2d(2),
            DSConvBlock(64, 64),
            DSConvBlock(64, 128),
        )
        # "gru": lightweight temporal head, keeps some sequence information
        # (phoneme ordering) instead of collapsing time and frequency
        # together in one shot. "pool": no extra params/compute, matches a
        # conventional DS-CNN classifier's global-average-pool.
        self.temporal = nn.GRU(input_size=128, hidden_size=128, batch_first=True) \
            if temporal_head == "gru" else None
        self.fc = nn.Linear(128, embed_dim)

    def forward(self, x):
        x = self.stem(x)
        x = self.ds_blocks(x)                    # (B, C, F, T)
        x = x.mean(dim=2)                         # average over freq -> (B, C, T)
        x = x.transpose(1, 2)                     # (B, T, C)
        if self.temporal_head == "gru":
            _, h = self.temporal(x)               # h: (1, B, 128)
            x = h.squeeze(0)
        else:
            x = x.mean(dim=1)                     # plain global-average-pool over time
        x = self.fc(x)
        return F.normalize(x, dim=-1)

class RepCNNEncoder(nn.Module):
    '''Placeholder -- implement the re-parameterizable multi-branch RepCNN
    backbone here when you're ready to upgrade. Must match DSCNNEncoder's
    exact interface: forward(x) with x=(batch,1,n_mels,time) ->
    (batch, EMBED_DIM) L2-normalized. Train it with the same Section 5-7
    pipeline below -- just swap BACKBONE = "repcnn" once this class is done.'''
    def __init__(self, n_mels=N_MELS, embed_dim=EMBED_DIM):
        super().__init__()
        raise NotImplementedError("Implement RepCNN here, matching DSCNNEncoder's interface")

def build_encoder(name=BACKBONE, **kw):
    if name == "dscnn":
        return DSCNNEncoder(**kw)
    elif name == "repcnn":
        return RepCNNEncoder(**kw)
    raise ValueError(f"Unknown backbone: {name}")

def param_count(module):
    return sum(p.numel() for p in module.parameters()) if module is not None else 0

encoder = build_encoder().to(device)
print(f"{BACKBONE} encoder ('{TEMPORAL_HEAD}' temporal head) parameter breakdown:")
if hasattr(encoder, "stem"):
    print(f"  stem:      {param_count(encoder.stem):>10,}")
    print(f"  ds_blocks: {param_count(encoder.ds_blocks):>10,}")
    print(f"  temporal:  {param_count(getattr(encoder, 'temporal', None)):>10,}  ({encoder.temporal_head})")
    print(f"  fc:        {param_count(encoder.fc):>10,}")
print(f"  TOTAL:     {param_count(encoder):>10,}")

## 5. Layer 1 data — Google Speech Commands (public, auto-downloaded)

35 keyword classes + a `_background_noise_` folder. This gives the encoder
broad exposure to "what other speech sounds like" — the negative-discrimination
signal — without you collecting a single clip yourself. **You run this section
once.** After `universal_encoder.pt` is saved, skip straight to Section 8
(enrollment) for every future wake word.

**Noise split fix:** the `_background_noise_` files are cut into non-overlapping
train/val/test *time chunks* per `NOISE_SPLIT_FRACTIONS` (config cell) — see
that comment for why a straight file-level split isn't workable with only a
handful of noise recordings, and why leaving this unfixed meant validation
noise was actually training noise, and the Section 10 test noise pool was
silently empty.

In [ ]:
# FIXED (v9.1): DATA_ROOT didn't exist yet as a directory -- torchaudio's
# downloader tries to open a temp file straight inside it without creating
# the folder first, which raised FileNotFoundError on a fresh environment
# (e.g. a new Colab runtime) rather than downloading anything.
os.makedirs(DATA_ROOT, exist_ok=True)

from torchaudio.datasets import SPEECHCOMMANDS

class SC(SPEECHCOMMANDS):
    def __init__(self, subset):
        super().__init__(root=DATA_ROOT, download=True, subset=subset)

train_set = SC("training")
val_set   = SC("validation")
test_set  = SC("testing")

# FIXED (v9.2): torchaudio's internal SPEECHCOMMANDS._walker format has
# changed across versions -- older releases store entries RELATIVE to
# dataset._path (so os.path.join(dataset._path, entry) is correct), but on
# at least one current version the entries are already fully resolved,
# which made the old unconditional join produce a duplicated path segment
# (".../speech_commands_v0.02/data/SpeechCommands/speech_commands_v0.02/...")
# and a FileNotFoundError/RuntimeError from the audio decoder. This resolves
# defensively instead of assuming either format, so it keeps working across
# torchaudio versions without needing another manual fix later.
def _resolve_walker_path(dataset, entry):
    joined = os.path.join(dataset._path, entry)
    if os.path.isfile(joined):
        return joined
    if os.path.isfile(entry):
        return entry
    raise FileNotFoundError(
        f"Could not resolve Speech Commands file from walker entry {entry!r} "
        f"(tried joined path {joined!r} and the entry as-is). torchaudio's "
        f"internal _walker path format may have changed again -- inspect "
        f"dataset._walker[0] and dataset._path directly to see the actual format."
    )

def group_by_label(dataset):
    groups = defaultdict(list)
    for i in range(len(dataset)):
        entry = dataset._walker[i]
        label = os.path.basename(os.path.dirname(entry))
        groups[label].append(_resolve_walker_path(dataset, entry))
    return groups

train_groups = group_by_label(train_set)
val_groups   = group_by_label(val_set)
test_groups  = group_by_label(test_set)

ALL_KEYWORD_CLASSES = sorted([c for c in train_groups if c != "_background_noise_"])
NOISE_FILES         = train_groups.get("_background_noise_", [])

MIN_TRAIN_CLIPS_PER_CLASS = N_SUPPORT + N_QUERY
KEYWORD_CLASSES = sorted([
    c for c in ALL_KEYWORD_CLASSES
    if len(train_groups[c]) >= MIN_TRAIN_CLIPS_PER_CLASS
])
excluded_train = sorted(set(ALL_KEYWORD_CLASSES) - set(KEYWORD_CLASSES))
if excluded_train:
    preview = excluded_train[:5]
    print(f"NOTE: {len(excluded_train)} classes have insufficient TRAINING data "
          f"(need >= {MIN_TRAIN_CLIPS_PER_CLASS}) and are excluded from episodic sampling: {preview}"
          + (" ..." if len(excluded_train) > 5 else ""))

print(f"keyword classes (train-qualified): {len(KEYWORD_CLASSES)} / {len(ALL_KEYWORD_CLASSES)}")
print(f"noise files: {len(NOISE_FILES)}")
assert len(KEYWORD_CLASSES) >= N_WAY, "Not enough classes for N_WAY episodes"

class ClipBank:
    def __init__(self, files):
        self.files = list(files)
        if not self.files:
            raise ValueError("ClipBank cannot be empty")

    def sample(self):
        return load_wav(random.choice(self.files))

    def sample_disjoint(self, n):
        if len(self.files) < n:
            raise ValueError(f"Need {n} distinct clips, found only {len(self.files)}")
        chosen = random.sample(self.files, n)
        return [load_wav(f) for f in chosen]

class SplitNoiseBank:
    """Background noise is outside Speech Commands' official subset split.
    Each source recording is cut into non-overlapping TIME chunks so train,
    model-validation, calibration/statistics, and final test never share audio.
    """
    def __init__(self, files, lo_frac, hi_frac, label):
        self.parts = []
        self.source_files = []
        self.label = label
        for f in files:
            wav = load_wav(f)
            n = wav.shape[0]
            lo, hi = int(n * lo_frac), int(n * hi_frac)
            part = wav[lo:hi]
            if part.shape[0] >= CLIP_LEN:
                self.parts.append(part)
                self.source_files.append(f)
        if not self.parts:
            raise ValueError(f"No noise segments long enough for '{label}' split ({lo_frac:.1%}-{hi_frac:.1%}).")

    @property
    def waves(self):
        return self.parts

    def sample(self):
        return random.choice(self.parts)

# Speech Commands background noise is manually split by TIME. Keep the final
# test chunk untouched until Section 10. The model-validation chunk is used
# for model validation; the separate public-speech statistics split below is
# disjoint at the CLIP level.
if NOISE_FILES:
    _f = NOISE_SPLIT_FRACTIONS
    noise_bank_train = SplitNoiseBank(NOISE_FILES, *_f["train"], "train")
    noise_bank_val   = SplitNoiseBank(NOISE_FILES, *_f["val"],   "val")
    noise_bank_calib = SplitNoiseBank(NOISE_FILES, *_f["calib"], "calib")
    noise_bank_test  = SplitNoiseBank(NOISE_FILES, *_f["test"],  "test")
    print(f"noise split -- train: {len(noise_bank_train.parts)} chunks, "
          f"model-val: {len(noise_bank_val.parts)} chunks, "
          f"calib: {len(noise_bank_calib.parts)} chunks, "
          f"test: {len(noise_bank_test.parts)} chunks")
else:
    noise_bank_train = noise_bank_val = noise_bank_calib = noise_bank_test = None
    print("WARNING: no background-noise files found; noise augmentation and noise testing are disabled.")

# Split the official validation utterances per class into THREE disjoint pools:
#   model_val  -> early stopping / backbone selection only
#   stats      -> PCA basis + public within-class variance prior only
#   calib      -> threshold calibration speech only
# The official test set remains completely untouched until Section 10.
PUBLIC_SPLIT_SEED = SEED + 701
MODEL_VAL_FRAC = 0.50
STATS_FRAC     = 0.25
CALIB_FRAC     = 0.25

val_model_groups, stats_groups, calib_groups = {}, {}, {}
for ci, c in enumerate(KEYWORD_CLASSES):
    files = list(val_groups.get(c, []))
    rng = random.Random(PUBLIC_SPLIT_SEED + ci)
    rng.shuffle(files)
    n = len(files)
    n_model = max(N_SUPPORT + N_QUERY, int(n * MODEL_VAL_FRAC))
    n_stats = max(2, int(n * STATS_FRAC))
    # Leave the remainder for calibration; enforce disjointness and ensure all
    # three pools are non-empty. If a class is unusually small, it is excluded
    # from model validation rather than backfilled from another split.
    if n_model + n_stats >= n:
        n_stats = max(2, n - n_model - 1)
    if n_model + n_stats >= n or n_stats < 2:
        continue
    val_model_groups[c] = files[:n_model]
    stats_groups[c] = files[n_model:n_model + n_stats]
    calib_groups[c] = files[n_model + n_stats:]

val_only_classes = sorted([
    c for c in KEYWORD_CLASSES
    if c in val_model_groups
    and len(val_model_groups[c]) >= N_SUPPORT + N_QUERY
    and len(stats_groups.get(c, [])) >= 2
    and len(calib_groups.get(c, [])) >= 2
])
assert len(val_only_classes) >= N_WAY, (
    "Not enough classes with sufficient disjoint model/stats/calibration public pools; "
    "lower N_WAY or N_SUPPORT+N_QUERY."
)

# FIXED (v7): train_class_banks was referenced in Section 6's training loop
# but never defined anywhere in the notebook -- a NameError bug that crashed
# the first call to build_episode(train_class_banks, ...) in Section 6.
train_class_banks = {c: ClipBank(train_groups[c]) for c in KEYWORD_CLASSES}

val_class_banks = {c: ClipBank(val_model_groups[c]) for c in val_only_classes}
stats_groups = {c: stats_groups[c] for c in val_only_classes}
calib_groups = {c: calib_groups[c] for c in val_only_classes}

print(f"train class banks: {len(train_class_banks)} classes (episodic training pool)")
print(f"public validation split: model={len(val_only_classes)} classes, "
      f"stats={sum(map(len, stats_groups.values()))} clips, "
      f"calibration={sum(map(len, calib_groups.values()))} clips; disjoint from model-val")


## 6. Episodic multi-class metric learning (the actual Layer-1 objective)

Each episode samples `N_WAY` random keyword classes, `N_SUPPORT` + `N_QUERY`
clips per class. The model learns "cluster same-word clips together, push
different-word clips apart" across many different words -- that generality is
what later lets a *frozen* encoder discriminate an unseen wake word from
unseen other-speech, without ever training on that specific word's negatives.

Support and query clips are drawn **disjoint** per class per episode
(`sample_disjoint`, Section 5) -- the same underlying recording never lands
in both sets within one episode, even under different random augmentation.

In [ ]:
def make_example(wav, noise_bank, p_noise=0.6, augment=True):
    wav = fix_length_random(wav)
    if augment:
        wav = time_shift(wav)
        wav = gain_jitter(wav)
        if noise_bank is not None and random.random() < p_noise:
            wav = mix_noise(wav, noise_bank.sample())
    return wav_to_features(wav).squeeze(0)   # (1, n_mels, time)

def build_episode(class_banks, noise_bank, n_way=N_WAY, n_support=N_SUPPORT, n_query=N_QUERY):
    classes = random.sample(list(class_banks.keys()), n_way)
    support, query, query_labels = [], [], []
    for ci, c in enumerate(classes):
        bank = class_banks[c]
        raw_clips = bank.sample_disjoint(n_support + n_query)
        support_clips, query_clips = raw_clips[:n_support], raw_clips[n_support:]
        for wav in support_clips:
            support.append(make_example(wav, noise_bank))
        for wav in query_clips:
            query.append(make_example(wav, noise_bank))
            query_labels.append(ci)
    support = torch.stack(support).view(n_way, n_support, 1, N_MELS, -1)
    query = torch.stack(query)
    query_labels = torch.tensor(query_labels, dtype=torch.long)
    return support, query, query_labels

def prototypical_loss(support, query, query_labels, encoder, n_way, n_support):
    B = support.shape[0]
    support_flat = support.view(n_way * n_support, 1, N_MELS, -1).to(device)
    s_embed = encoder(support_flat).view(n_way, n_support, -1)
    prototypes = s_embed.mean(dim=1)                       # (n_way, embed_dim)

    q_embed = encoder(query.to(device))                    # (n_way*n_query, embed_dim)
    dists = torch.cdist(q_embed, prototypes) ** 2
    logits = -dists
    labels = query_labels.to(device)
    loss = F.cross_entropy(logits, labels)
    acc = (logits.argmax(dim=-1) == labels).float().mean().item()
    return loss, acc

In [ ]:
def _rng_state():
    state = {"random": random.getstate(), "numpy": np.random.get_state(),
              "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state

def _restore_rng_state(state):
    random.setstate(state["random"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if "torch_cuda" in state and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["torch_cuda"])

def _ckpt_payload(epoch, best_val_acc, no_improve):
    return {"encoder_state": encoder.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "best_val_acc": best_val_acc,
            "no_improve": no_improve,
            "rng_state": _rng_state(),
            "backbone": BACKBONE,
            "temporal_head": TEMPORAL_HEAD,
            "config": {"SR": SR, "N_MELS": N_MELS, "EMBED_DIM": EMBED_DIM}}

optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)
start_epoch, best_val_acc, no_improve = 1, -1.0, 0

if RESUME_TRAINING and os.path.exists(RESUME_CKPT):
    ckpt = torch.load(RESUME_CKPT, map_location=device)
    encoder.load_state_dict(ckpt["encoder_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    no_improve = ckpt["no_improve"]
    if "rng_state" in ckpt:
        _restore_rng_state(ckpt["rng_state"])
        print(f"Resumed from epoch {ckpt['epoch']}, best_val_acc so far: {best_val_acc:.3f} "
              f"(RNG state restored -- episode sampling continues reproducibly).")
    else:
        print(f"Resumed from epoch {ckpt['epoch']}, best_val_acc so far: {best_val_acc:.3f} "
              f"(no RNG state in this checkpoint -- episode sampling will diverge from "
              f"an uninterrupted run, training is still valid).")
elif RESUME_TRAINING:
    print(f"RESUME_TRAINING=True but {RESUME_CKPT} not found -- starting fresh.")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    encoder.train()
    tr_losses, tr_accs = [], []
    for i in range(EPISODES_PER_EPOCH):
        support, query, qlabels = build_episode(train_class_banks, noise_bank_train)
        loss, acc = prototypical_loss(support, query, qlabels, encoder, N_WAY, N_SUPPORT)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tr_losses.append(loss.item()); tr_accs.append(acc)

    encoder.eval()
    va_losses, va_accs = [], []
    with torch.no_grad():
        for _ in range(VAL_EPISODES):
            support, query, qlabels = build_episode(val_class_banks, noise_bank_val)  # FIXED: was noise_bank_train -- see Section 5 note
            loss, acc = prototypical_loss(support, query, qlabels, encoder, N_WAY, N_SUPPORT)
            va_losses.append(loss.item()); va_accs.append(acc)

    tr_acc, va_acc = np.mean(tr_accs), np.mean(va_accs)
    tr_loss, va_loss = float(np.mean(tr_losses)), float(np.mean(va_losses))
    improved = va_acc > best_val_acc
    print(f"epoch {epoch:02d} | train_acc {tr_acc:.3f} | val_acc {va_acc:.3f}" + ("  <- best" if improved else ""))

    writer.add_scalars("layer1/accuracy", {"train": tr_acc, "val": va_acc}, epoch)
    writer.add_scalars("layer1/loss", {"train": tr_loss, "val": va_loss}, epoch)
    writer.flush()

    if improved:
        best_val_acc = va_acc; no_improve = 0
        torch.save(_ckpt_payload(epoch, best_val_acc, no_improve), UNIVERSAL_CKPT)
    else:
        no_improve += 1

    # Always checkpoint the latest state (regardless of improvement) to the
    # resume path -- a Colab disconnect mid-run loses at most the current
    # epoch, not the whole run. UNIVERSAL_CKPT above stays "best model only".
    torch.save(_ckpt_payload(epoch, best_val_acc, no_improve), RESUME_CKPT)

    if no_improve >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break

print(f"Best val episode accuracy: {best_val_acc:.3f}")
print(f"Universal encoder saved to {UNIVERSAL_CKPT} -- this is Layer 1. Reuse it from here on.")

## 7. Freeze Layer 1

Run this cell (and everything above skipped) any time you're starting fresh
with an already-trained `universal_encoder.pt`. From here down is the only
part you touch per wake word.

In [ ]:
ckpt = torch.load(UNIVERSAL_CKPT, map_location=device)
encoder = build_encoder(ckpt["backbone"], temporal_head=ckpt.get("temporal_head", "gru")).to(device)
encoder.load_state_dict(ckpt["encoder_state"])
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False
print(f"Layer 1 encoder loaded and frozen (backbone={ckpt['backbone']}, "
      f"temporal_head={ckpt.get('temporal_head', 'gru')}).")

## 7.5 Shared statistics basis — conservative positive-only enrollment

Layer 2 only has a handful of genuine enrollment recordings. The default `pca_diag`
mode therefore does **not** estimate a full covariance from those clips. Instead,
this section learns a compact basis from held-out public Speech Commands data,
using within-class residuals so the basis represents acoustic variation inside
words rather than simply separating word identities.

The enrollment stage then fits only one variance per compact dimension and shrinks
that variance toward a public-data prior. This is deliberately conservative for
3–5 enrollment clips.


In [ ]:
def embed_wav(encoder, wav):
    feat = wav_to_features(fix_length_center(wav))
    with torch.no_grad():
        return encoder(feat.to(device)).cpu().numpy().squeeze(0)

def fit_pca_basis(encoder, n_samples=1600, n_components=PCA_DIM, seed=SEED):
    rng = random.Random(seed)
    residuals = []
    per_class = max(2, n_samples // max(1, len(stats_groups)))
    for ci, c in enumerate(sorted(stats_groups)):
        files = list(stats_groups[c])
        rng.shuffle(files)
        chosen = files[:per_class]
        if len(chosen) < 2:
            continue
        arr = np.stack([embed_wav(encoder, load_wav(f)) for f in chosen])
        residuals.append(arr - arr.mean(axis=0, keepdims=True))

    if not residuals:
        raise RuntimeError("Not enough disjoint public statistics data to build PCA basis.")
    residuals = np.concatenate(residuals, axis=0)
    max_components = min(residuals.shape[1], residuals.shape[0] - 1)
    if n_components > max_components:
        raise RuntimeError(f"PCA_DIM={n_components} exceeds available rank {max_components} from public statistics data.")

    _, S, Vt = np.linalg.svd(residuals, full_matrices=False)
    components = Vt[:n_components]
    projected = residuals @ components.T
    prior_var = np.clip(np.var(projected, axis=0, ddof=1), MIN_VARIANCE, None)
    explained = float((S[:n_components] ** 2).sum() / max((S ** 2).sum(), 1e-12))
    print(f"PCA basis: {n_components} within-class dimensions from {len(residuals)} DISJOINT public statistics residuals; "
          f"explained within-class variance={explained:.1%}")
    return {
        "mean": np.zeros(EMBED_DIM, dtype=np.float32).tolist(),
        "components": components.astype(np.float32).tolist(),
        "prior_var": prior_var.astype(np.float32).tolist(),
        "n_components": int(n_components),
        "basis_samples": int(len(residuals)),
        "basis_source": "public_validation_stats_split_only",
    }

pca_basis = None
if COV_MODE in ("pca_diag", "pca"):
    pca_basis = fit_pca_basis(encoder)
else:
    print(f"COV_MODE='{COV_MODE}' -- no PCA basis required.")

def _project(z, cov_mode, pca_basis):
    z = np.asarray(z)
    if cov_mode in ("pca_diag", "pca"):
        return (z - np.asarray(pca_basis.get("mean", np.zeros(z.shape[-1])))) @ np.asarray(pca_basis["components"]).T
    return z

def _public_prior_variance(cov_mode, pca_basis, dim):
    if cov_mode == "pca_diag" and pca_basis is not None:
        return np.clip(np.asarray(pca_basis["prior_var"], dtype=float), MIN_VARIANCE, None)
    return np.ones(dim, dtype=float)

def _fit_profile_stats_from_clip_groups(clip_variant_embeds, cov_mode=COV_MODE, pca_basis=pca_basis):
    groups = [np.stack([_project(e, cov_mode, pca_basis) for e in clip]) for clip in clip_variant_embeds]
    clip_means = np.stack([g.mean(axis=0) for g in groups])
    mean = clip_means.mean(axis=0)

    within = []
    for g in groups:
        within.append(g.var(axis=0, ddof=1) if g.shape[0] > 1 else np.zeros(g.shape[1]))
    within_var = np.mean(np.stack(within), axis=0)
    between_var = clip_means.var(axis=0, ddof=1) if clip_means.shape[0] > 1 else np.zeros(clip_means.shape[1])
    raw_var = between_var + within_var

    if cov_mode in ("diagonal", "pca_diag"):
        prior = _public_prior_variance(cov_mode, pca_basis, raw_var.shape[0])
        var_reg = np.maximum((1-COV_SHRINKAGE)*raw_var + COV_SHRINKAGE*prior, MIN_VARIANCE)
        return mean, np.diag(1.0 / var_reg)

    all_proj = np.concatenate(groups, axis=0)
    cov = np.cov(all_proj.T) if all_proj.shape[0] > 1 else np.eye(all_proj.shape[1]) * MIN_VARIANCE
    target = np.eye(cov.shape[0]) * max(float(np.trace(cov) / cov.shape[0]), MIN_VARIANCE)
    cov_reg = (1-COV_SHRINKAGE)*cov + COV_SHRINKAGE*target
    return mean, np.linalg.pinv(cov_reg)

def _fit_profile_stats(embeds, shrinkage=COV_SHRINKAGE, cov_mode=COV_MODE, pca_basis=pca_basis):
    proj = np.stack([_project(e, cov_mode, pca_basis) for e in embeds])
    mean = proj.mean(axis=0)
    if cov_mode in ("diagonal", "pca_diag"):
        var = proj.var(axis=0, ddof=1) if proj.shape[0] > 1 else np.zeros(proj.shape[1])
        prior = _public_prior_variance(cov_mode, pca_basis, proj.shape[1])
        var_reg = np.maximum((1-shrinkage)*var + shrinkage*prior, MIN_VARIANCE)
        cov_inv = np.diag(1.0 / var_reg)
    else:
        cov = np.cov(proj.T) if proj.shape[0] > 1 else np.eye(proj.shape[1]) * MIN_VARIANCE
        target = np.eye(cov.shape[0]) * max(float(np.trace(cov) / cov.shape[0]), MIN_VARIANCE)
        cov_reg = (1-shrinkage)*cov + shrinkage*target
        cov_inv = np.linalg.pinv(cov_reg)
    return mean, cov_inv

def mahalanobis_score(z, profile):
    zp = _project(z, profile["cov_mode"], profile.get("pca_basis"))
    diff = zp - np.asarray(profile["mean"])
    return float(diff @ np.asarray(profile["cov_inv"]) @ diff.T)


## 8. Layer 2 — Enrollment (positive samples only, this is your workflow)

Put 3-5 recordings of your wake word in `ENROLLMENT_DIR`. Per our earlier
discussion, vary them: one close to the mic, one farther away, one quieter,
one at a slightly different pace, one with mild background noise. Identical
clean repeats teach the profile nothing about your real-world variation.

Uses `fix_length_center` (never random crop) so the full word is always
captured intact -- no truncated positives.

Each raw clip is expanded into `N_ENROLL_AUGS` mildly-augmented variants
(small gain/timing jitter only -- not synthesized new content) before fitting
the mean/covariance, so the covariance estimate has more support without
needing more real recordings from you. Variants of the same raw clip are
kept grouped, not flattened -- this matters for honest self-scoring below.

**Be realistic about what augmentation buys you:** 5 raw clips x 5 variants
each is not 25 independent observations -- the variants of one clip are
correlated with each other and with that clip, so you're still fundamentally
working with about 5 independent pieces of evidence. `COV_MODE` (config
cell) is what actually controls how much statistical risk that implies —
see the config comment for the tradeoffs.

In [ ]:
def enroll(encoder, files, n_augs=N_ENROLL_AUGS, cov_mode=COV_MODE, pca_basis=pca_basis):
    # clip_variant_embeds[i] = [unaugmented_embed, aug1_embed, aug2_embed, ...]
    # for raw clip i -- kept grouped rather than flattened into one pool.
    clip_variant_embeds = []
    for f in files:
        wav = fix_length_center(load_wav(f))
        variants = [wav]
        for _ in range(n_augs):
            v = gain_jitter(wav, db_range=(-3, 3))
            v = time_shift(v, max_shift=0.05)
            variants.append(v)
        embeds_for_clip = [embed_wav(encoder, v) for v in variants]
        clip_variant_embeds.append(embeds_for_clip)

    all_embeds = [e for clip in clip_variant_embeds for e in clip]
    raw_embeds = [clip[0] for clip in clip_variant_embeds]

    mean, cov_inv = _fit_profile_stats_from_clip_groups(clip_variant_embeds, cov_mode=cov_mode, pca_basis=pca_basis)

    return {
        "mean": mean.tolist(), "cov_inv": cov_inv.tolist(),
        "cov_mode": cov_mode, "pca_basis": pca_basis,
        "n_clips": len(files), "n_total_embeds": len(all_embeds),
        "raw_embeds": [e.tolist() for e in raw_embeds],
        "clip_variant_embeds": [[e.tolist() for e in clip] for clip in clip_variant_embeds],
    }

enrollment_files = sorted(glob.glob(os.path.join(ENROLLMENT_DIR, "*.wav")))
print(f"enrollment clips found: {len(enrollment_files)}")
assert len(enrollment_files) >= 3, "Record at least 3 clips into ENROLLMENT_DIR before running this."

profile = enroll(encoder, enrollment_files)
print(f"Enrollment profile built from {profile['n_clips']} raw clips "
      f"({profile['n_total_embeds']} total embeddings after augmentation), "
      f"cov_mode='{profile['cov_mode']}'.")

## 9. Threshold calibration

Zero *extra* collection here -- this reuses the same public Speech Commands
val set and noise bank from Section 5 (already downloaded) purely as a
sanity check for picking a sensible default threshold. It is not
per-wake-word negative collection; it's the same one-time public dataset
doing double duty.

Fixes versus a naive single-percentile approach:
1. **Continuous listening needs a much tighter FAR than a one-shot
   classifier.** `TARGET_FAR` (config cell) is set far lower than a typical
   1% classifier target, because this threshold is evaluated against a live
   stream scored ~10x/second, not a single decision.
2. **Speech and noise are calibrated separately, then combined
   conservatively.** Blending both populations into one percentile lets
   whichever is more numerous or naturally tighter dominate the estimate,
   silently under-protecting against the other. Each gets its own threshold;
   the final threshold is the **stricter (lower) of the two**.
3. **Non-negative, right-skewed scores get a Gamma-tail fallback, not a
   Gaussian one.** A Gamma distribution (fit by method of moments) respects
   the non-negative support and right skew that squared-Mahalanobis-style
   scores actually have.
4. **Self-scores use leave-one-*clip*-out, not leave-one-sample-out**,
   since each raw clip now expands into several correlated augmented
   variants (Section 8) -- leaving out one *sample* would still leave its
   highly correlated siblings in the profile.
5. **Noise calibration now genuinely uses held-out noise** (`noise_bank_val`
   from Section 5's time-split), not training noise relabeled as validation
   -- see the Section 5 note for why the previous version's fallback here
   was silently pulling from the training pool.

**Honesty check on the number this produces:** with `TARGET_FAR = 1e-4` and
this few calibration samples, the threshold below is a Gamma-tail
*extrapolation* targeting that rate -- not a directly measured one. Treat it
as "selected under a model targeting 1e-4," and look to Section 10's
held-out test for the actual measured FAR/FRR at that threshold.

In [ ]:
from scipy import stats

def score_files(encoder, files):
    scores = []
    for f in files:
        z = embed_wav(encoder, load_wav(f))
        scores.append(mahalanobis_score(z, profile))
    return np.array(scores)

def score_noise_bank(encoder, bank, n_windows=300):
    """Use non-overlapping 1s windows within one held-out noise split."""
    if bank is None:
        return np.array([])
    candidates = []
    for wav in bank.waves:
        for start in range(0, max(0, wav.shape[0] - CLIP_LEN + 1), CLIP_LEN):
            candidates.append(wav[start:start+CLIP_LEN])
    if not candidates:
        return np.array([])
    rng = random.Random(SEED + len(candidates) + n_windows)
    rng.shuffle(candidates)
    scores=[]
    for seg in candidates[:min(n_windows, len(candidates))]:
        scores.append(mahalanobis_score(embed_wav(encoder, seg), profile))
    return np.asarray(scores)

def leave_one_clip_out_self_scores(clip_variant_embeds, cov_mode, pca_basis):
    '''Score each raw clip's UNAUGMENTED embedding against a mean/covariance
    built from every OTHER clip's variants (excluding ALL variants of the
    held-out clip, not just one sample of it -- augmented siblings are
    highly correlated, so leaving only a single sample out would still leak
    information about that clip into the profile).'''
    n_clips = len(clip_variant_embeds)
    out = []
    for i in range(n_clips):
        other_embeds = [e for j, clip in enumerate(clip_variant_embeds)
                         if j != i for e in clip]
        mean, cov_inv = _fit_profile_stats(other_embeds, cov_mode=cov_mode, pca_basis=pca_basis)
        zp = _project(clip_variant_embeds[i][0], cov_mode, pca_basis)
        diff = zp - mean
        out.append(float(diff @ cov_inv @ diff.T))
    return np.array(out)

def gamma_tail_extrapolate(scores, target_far):
    '''Non-negative, right-skewed scores (like squared Mahalanobis
    distances) are poorly modeled by a Gaussian tail, which can extrapolate
    into physically impossible negative thresholds. Gamma, fit by method of
    moments, respects the non-negative support and the right skew.'''
    mean = float(np.mean(scores))
    var = max(float(np.var(scores)), 1e-6)
    shape, scale = mean ** 2 / var, var / mean
    return max(float(stats.gamma.ppf(target_far, a=shape, scale=scale)), 0.0)

def calibrate_threshold(scores, target_far, label, min_samples_per_tail=20):
    n_needed = min_samples_per_tail / target_far
    if len(scores) >= n_needed:
        return float(np.percentile(scores, target_far * 100)), "empirical"
    empirical_est = float(np.percentile(scores, target_far * 100))
    extrapolated = gamma_tail_extrapolate(scores, target_far)
    print(f"NOTE [{label}]: {len(scores)} calibration samples is below the "
          f"~{int(n_needed):,} needed to trust an empirical {target_far:.1e} "
          f"percentile directly. Using a Gamma-tail extrapolation instead: "
          f"{extrapolated:.2f} (raw empirical percentile would have been "
          f"{empirical_est:.2f}, not trusted here).")
    return extrapolated, "extrapolated (gamma)"

# Full class pool (not a random subset), speech and noise calibrated separately
other_speech_files = []
for ci, c in enumerate(sorted(calib_groups)):
    files_c = list(calib_groups[c])
    random.Random(SEED + 1100 + ci).shuffle(files_c)
    other_speech_files += files_c[:40]

speech_scores = score_files(encoder, other_speech_files)
noise_scores = score_noise_bank(encoder, noise_bank_calib, n_windows=300) if noise_bank_calib else np.array([])

print(f"calibration pool: {len(speech_scores)} other-speech windows from DISJOINT public calibration split, "
      f"{len(noise_scores)} noise windows from DISJOINT noise_bank_calib")

speech_threshold, speech_method = calibrate_threshold(speech_scores, TARGET_FAR, "speech")
if len(noise_scores) > 0:
    noise_threshold, noise_method = calibrate_threshold(noise_scores, TARGET_FAR, "noise")
else:
    noise_threshold, noise_method = float("inf"), "n/a (no noise samples)"

# Stricter (lower) of the two population-specific thresholds.
threshold = min(speech_threshold, noise_threshold)
method = (f"speech={speech_method}({speech_threshold:.2f}), "
          f"noise={noise_method}({noise_threshold:.2f}), min taken")

self_scores = leave_one_clip_out_self_scores(
    profile["clip_variant_embeds"], profile["cov_mode"], profile.get("pca_basis"))

print(f"leave-one-clip-out self scores -- mean {self_scores.mean():.2f}, max {self_scores.max():.2f} "
      f"(honest estimate of a genuinely new utterance, not the clips used to build the profile)")
print(f"target-FAR threshold: {threshold:.2f}  [{method}]")

# FIXED (v9): the previous version printed this warning and then saved the
# too-tight threshold ANYWAY. With only 3-5 real enrollment clips at
# TARGET_FAR=1e-4, this fires on nearly every run -- and once it does, even
# your OWN genuine wake-word utterances score above the saved threshold, so
# WakeWordTrigger's `score < threshold` check can never pass. The detector
# doesn't get worse in that state -- it becomes mathematically incapable of
# ever firing, for anyone. This is almost certainly why detection appeared
# to "do nothing" despite a threshold being computed and saved.
target_far_threshold = threshold
RELAX_MARGIN = 1.25  # safety margin above the worst leave-one-clip-out self-score
usable_floor = float(self_scores.max()) * RELAX_MARGIN

if target_far_threshold < usable_floor:
    threshold = usable_floor
    threshold_source = "relaxed-for-usability"
    print(f"NOTE: target-FAR threshold ({target_far_threshold:.2f}) is tighter than this "
          f"profile's own genuine-sample spread -- using it as-is would make the detector "
          f"unable to fire even on your own voice. Relaxing to {RELAX_MARGIN:.2f}x the worst "
          f"leave-one-clip-out self-score instead: {threshold:.2f}.")
    print("      This means FAR is no longer guaranteed to be near TARGET_FAR -- Section 10's "
          "MEASURED FAR/FRR on held-out data (not this target) is the number to trust. If FAR "
          "there is too high for comfort, the real fix is more/more-varied enrollment clips "
          "(distance/volume/pace/background) so the target-FAR threshold stops being tighter "
          "than your own natural variation, not loosening TARGET_FAR further.")
else:
    threshold_source = "target-FAR calibration"
    print(f"chosen threshold: {threshold:.2f}  [{method}] -- within the enrollment spread, used as-is.")

profile["threshold"] = threshold
profile["target_far_threshold"] = target_far_threshold
profile["threshold_source"] = threshold_source
with open(PROFILE_PATH, "w") as f:
    json.dump(profile, f)
print("Saved profile + threshold to", PROFILE_PATH)

writer.add_scalar("layer2/target_far_threshold", target_far_threshold, 0)
writer.add_scalar("layer2/used_threshold", threshold, 0)
writer.add_text("layer2/threshold_source", threshold_source, 0)
writer.flush()

## 10. Held-out evaluation — EER, FAR, FRR

Impostor data uses the Speech Commands **testing** split (never touched by
training, validation, or calibration) for speech, plus `noise_bank_test`
(the time-sliced test chunk of the noise files, Section 5) for noise --
**fixed**: the previous version looked for `_background_noise_` inside the
testing split directly, which doesn't exist there (same root cause as the
validation-noise issue), so the noise half of this evaluation was silently
running on zero samples with no warning printed.

**Genuine data — two modes:**
- If `POSITIVE_EVAL_DIR` (config cell) contains recordings, they're used
  directly as genuine scores. This is the trustworthy case: real, unseen
  wake-word attempts, ideally recorded in different conditions than your
  enrollment clips.
- If that folder is empty/missing, this falls back to the same
  leave-one-clip-out scores from Section 9 -- still useful as a rough
  proxy, but it's testing "held-out variants of the same handful of
  recordings," not real held-out attempts. Treat FRR from this fallback as
  directional, not certified.

In [ ]:
positive_eval_files = (sorted(glob.glob(os.path.join(POSITIVE_EVAL_DIR, "*.wav")))
                        if POSITIVE_EVAL_DIR else [])

if positive_eval_files and len(positive_eval_files) < 20:
    print(f"WARNING: only {len(positive_eval_files)} fresh positive evaluation clips found; FRR will be noisy.")

if positive_eval_files:
    genuine_scores = score_files(encoder, positive_eval_files)
    genuine_source = f"{len(positive_eval_files)} held-out recordings in POSITIVE_EVAL_DIR"
    print(f"Using {genuine_source} for genuine scores -- this is the trustworthy case.")
else:
    genuine_scores = self_scores
    genuine_source = f"{len(self_scores)} leave-one-clip-out self-scores (POSITIVE_EVAL_DIR empty/missing)"
    print(f"NOTE: using {genuine_source}. For a real FRR estimate, record 20-100 more "
          f"wake-word clips (different room/distance/day/mic) into POSITIVE_EVAL_DIR and "
          f"rerun this section -- leave-one-clip-out is a rough proxy, not measured generalization.")


test_other_files = []
for ci, c in enumerate(KEYWORD_CLASSES):
    files_c = list(test_groups.get(c, []))
    random.Random(SEED + 2200 + ci).shuffle(files_c)
    test_other_files += files_c[:40]

test_speech_scores = score_files(encoder, test_other_files)
# FIXED: test_groups never contains "_background_noise_" (same root cause as
# the validation-noise issue) -- the old `test_groups.get("_background_noise_", [])`
# was always []. noise_bank_test (Section 5's time-sliced test chunk) now
# gives this a real, held-out noise-impostor pool instead of silently skipping it.
test_noise_scores = score_noise_bank(encoder, noise_bank_test, n_windows=300) if noise_bank_test else np.array([])

impostor_scores = np.concatenate([test_speech_scores, test_noise_scores]) \
    if len(test_noise_scores) else test_speech_scores

def far_frr_at(genuine, impostor, th):
    far = float((impostor < th).mean())   # impostor incorrectly accepted
    frr = float((genuine >= th).mean())   # genuine incorrectly rejected
    return far, frr

def compute_eer(genuine, impostor, n_points=500):
    lo, hi = min(genuine.min(), impostor.min()), max(genuine.max(), impostor.max())
    best = None
    for th in np.linspace(lo, hi, n_points):
        far, frr = far_frr_at(genuine, impostor, th)
        gap = abs(far - frr)
        if best is None or gap < best[0]:
            best = (gap, (far + frr) / 2, th)
    return best[1], best[2]

far_op, frr_op = far_frr_at(genuine_scores, impostor_scores, threshold)
eer, eer_threshold = compute_eer(genuine_scores, impostor_scores)

if len(impostor_scores) < 500:
    print(f"WARNING: only {len(impostor_scores)} held-out impostor windows; FAR estimates will be noisy.")

print(f"held-out test pool: {len(impostor_scores)} impostor windows "
      f"({len(test_speech_scores)} speech + {len(test_noise_scores)} noise), "
      f"{len(genuine_scores)} genuine samples ({genuine_source})")
print(f"at calibrated threshold ({threshold:.2f}): FAR={far_op:.4%}, FRR={frr_op:.4%}")
print(f"Equal Error Rate: {eer:.4%} at threshold {eer_threshold:.2f}")
if not positive_eval_files:
    print("NOTE: FRR is estimated from leave-one-clip-out self-scores, not fresh "
          "recordings -- treat it as directional until validated with POSITIVE_EVAL_DIR data.")

writer.add_scalar("eval/FAR_at_operating_threshold", far_op, 0)
writer.add_scalar("eval/FRR_at_operating_threshold", frr_op, 0)
writer.add_scalar("eval/EER", eer, 0)
writer.add_scalar("eval/n_impostor_windows", len(impostor_scores), 0)
writer.add_scalar("eval/n_genuine_samples", len(genuine_scores), 0)
writer.flush()

### 10.5 Visualizing the calibration (score distributions + DET curve)

Every number in Section 10 (threshold, FAR, FRR, EER) comes from one held-out score
distribution comparison. A single scalar EER can hide a lot -- e.g. a bimodal impostor
distribution, or a genuine distribution with a heavy tail from one bad enrollment clip.
Plotting the actual distributions and the DET curve makes those situations visible instead
of silently averaged away.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Left: score histograms -------------------------------------------------
ax = axes[0]
bins = np.linspace(
    0, max(float(genuine_scores.max()), float(impostor_scores.max()), 1.0), 40
)
ax.hist(impostor_scores, bins=bins, alpha=0.55, label=f"impostor (n={len(impostor_scores)})", color="tab:red")
ax.hist(genuine_scores, bins=bins, alpha=0.55, label=f"genuine (n={len(genuine_scores)}, {genuine_source.split(' ')[0]}...)", color="tab:blue")
ax.axvline(threshold, color="black", linestyle="--", linewidth=1.5, label=f"operating threshold ({threshold:.2f})")
ax.axvline(eer_threshold, color="grey", linestyle=":", linewidth=1.5, label=f"EER threshold ({eer_threshold:.2f})")
ax.set_xlabel("Mahalanobis score (lower = more wake-word-like)")
ax.set_ylabel("count")
ax.set_title("Genuine vs. impostor score distributions")
ax.legend(fontsize=8)

# --- Right: DET curve (FAR vs FRR swept over threshold) --------------------
ax = axes[1]
lo = min(genuine_scores.min(), impostor_scores.min())
hi = max(genuine_scores.max(), impostor_scores.max())
ths = np.linspace(lo, hi, 300)
fars = np.array([far_frr_at(genuine_scores, impostor_scores, t)[0] for t in ths])
frrs = np.array([far_frr_at(genuine_scores, impostor_scores, t)[1] for t in ths])
ax.plot(fars, frrs, color="tab:purple")
ax.scatter([far_op], [frr_op], color="black", zorder=5, label=f"operating point (FAR={far_op:.2%}, FRR={frr_op:.2%})")
ax.scatter([eer], [eer], color="grey", marker="x", zorder=5, label=f"EER={eer:.2%}")
ax.plot([0, 1], [0, 1], color="lightgrey", linestyle="--", linewidth=1)
ax.set_xlabel("False Accept Rate")
ax.set_ylabel("False Reject Rate")
ax.set_title("DET curve (held-out test split)")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("calibration_report.png", dpi=130)
plt.show()
print("Saved calibration_report.png -- worth keeping alongside wakeword_profile.json as a")
print("record of what this profile's operating point actually looked like at export time.")

writer.add_figure("eval/score_distributions_and_DET_curve", fig, 0)
writer.flush()


## 11. Temporal confirmation + debounce/hangover trigger logic

With a 1s window and a 0.1s scoring hop, consecutive windows overlap by 90%
-- nearly the same audio scored three times, not three independent
observations. Confirmations must be spaced apart in real time to count as
separate evidence.

**Fixed:** the previous version enforced spacing by only feeding every 3rd
*VAD-positive* score into the confirmation buffer (`call_count % 3`). That
approximates ~0.3s spacing only when the VAD accepts frames at a steady
rate -- gaps from VAD rejections (silence, or the VAD flipping on
loud-but-non-speech noise) meant the *actual* elapsed time between counted
confirmations could drift well away from 0.3s without anything catching it.
`WakeWordTrigger.update` now takes the real stream timestamp on every call
and enforces `MIN_CONFIRM_SPACING_SEC` directly against elapsed time, so the
spacing guarantee holds regardless of VAD behavior in between.

In [ ]:
class WakeWordTrigger:
    def __init__(self, threshold, confirm_windows=CONFIRM_WINDOWS,
                 hangover_sec=HANGOVER_SEC,
                 min_confirm_spacing_sec=MIN_CONFIRM_SPACING_SEC):
        self.threshold = threshold
        self.confirm_windows = confirm_windows
        self.hangover_sec = hangover_sec
        self.min_confirm_spacing_sec = min_confirm_spacing_sec
        self.confirm_times = []          # real timestamps of counted (spaced-out) hits
        self.last_trigger_t = -1e9

    def update(self, score, t):
        '''t is the actual stream timestamp in seconds -- NOT a call counter.
        Spacing is enforced against elapsed time directly, so a run of
        VAD-rejected frames in between can't distort how far apart counted
        confirmations really are (a real bug in the previous call-count-based
        version -- see Section 11).'''
        is_hit = score < self.threshold
        if is_hit:
            if not self.confirm_times or (t - self.confirm_times[-1]) >= self.min_confirm_spacing_sec:
                self.confirm_times.append(t)

        # Drop confirmations that have aged out -- keeps the buffer scoped to
        # a single plausible utterance rather than accumulating hits across
        # unrelated moments far apart in time.
        max_span = self.min_confirm_spacing_sec * max(1, self.confirm_windows - 1) + 0.5
        self.confirm_times = [ct for ct in self.confirm_times if t - ct <= max_span]

        confirmed = len(self.confirm_times) >= self.confirm_windows
        cooled_down = (t - self.last_trigger_t) > self.hangover_sec
        if confirmed and cooled_down:
            self.last_trigger_t = t
            self.confirm_times = []
            return True
        return False

### 11.5 Sanity tests for `WakeWordTrigger`

The v7 changelog fixed a real timestamp-spacing bug in this class. Bugs like that are easy
to reintroduce silently (e.g. while tuning `CONFIRM_WINDOWS`/`HANGOVER_SEC`), and the class
has no automated coverage anywhere else in the notebook. These are plain `assert`-based
checks, not a full test framework -- just enough to catch a regression before it reaches a
live deployment.

In [ ]:
def _test_trigger_basic_confirmation():
    trig = WakeWordTrigger(threshold=1.0, confirm_windows=3, hangover_sec=1.0, min_confirm_spacing_sec=0.3)
    # Three hits spaced exactly at the minimum spacing should confirm on the third.
    assert trig.update(0.5, t=0.0) is False
    assert trig.update(0.5, t=0.3) is False
    assert trig.update(0.5, t=0.6) is True, "expected trigger on the 3rd properly-spaced hit"

def _test_trigger_rejects_too_close_hits():
    trig = WakeWordTrigger(threshold=1.0, confirm_windows=3, hangover_sec=1.0, min_confirm_spacing_sec=0.3)
    # Hits closer together than min spacing must NOT count as separate confirmations,
    # regardless of how many raw calls occur in between.
    assert trig.update(0.5, t=0.00) is False
    assert trig.update(0.5, t=0.05) is False   # too close -> not counted
    assert trig.update(0.5, t=0.10) is False   # still too close -> not counted
    assert trig.update(0.5, t=0.35) is False   # counted (2nd real confirmation)
    assert len(trig.confirm_times) == 2, f"expected 2 counted confirmations, got {len(trig.confirm_times)}"

def _test_trigger_spacing_survives_vad_gaps():
    # Regression test for the exact v6->v7 bug: a long run of VAD-rejected frames between
    # counted hits must not let the *elapsed* spacing silently drift under the minimum.
    trig = WakeWordTrigger(threshold=1.0, confirm_windows=2, hangover_sec=1.0, min_confirm_spacing_sec=0.3)
    assert trig.update(0.5, t=0.0) is False
    # Simulate a gap that's shorter than min spacing even though "many calls" could have
    # happened in between in call-count terms -- this must still be rejected.
    assert trig.update(0.5, t=0.1) is False
    assert len(trig.confirm_times) == 1, "hit at t=0.1 should have been rejected (too close in real time)"

def _test_trigger_hangover_blocks_immediate_retrigger():
    trig = WakeWordTrigger(threshold=1.0, confirm_windows=2, hangover_sec=1.0, min_confirm_spacing_sec=0.3)
    assert trig.update(0.5, t=0.0) is False
    assert trig.update(0.5, t=0.3) is True
    # Immediately after a trigger, even valid confirmations shouldn't fire again
    # until hangover_sec has elapsed.
    assert trig.update(0.5, t=0.6) is False
    assert trig.update(0.5, t=0.9) is False
    assert trig.update(0.5, t=1.4) is True, "expected a new trigger once hangover has elapsed"

def _test_trigger_stale_confirmations_expire():
    trig = WakeWordTrigger(threshold=1.0, confirm_windows=3, hangover_sec=1.0, min_confirm_spacing_sec=0.3)
    assert trig.update(0.5, t=0.0) is False
    assert trig.update(0.5, t=0.3) is False
    # Long gap -- the first two hits are stale evidence of a DIFFERENT utterance and
    # must not combine with a hit much later to fire a trigger.
    assert trig.update(0.5, t=10.0) is False
    assert len(trig.confirm_times) == 1, "old confirmations should have aged out of the buffer"

for fn in [_test_trigger_basic_confirmation, _test_trigger_rejects_too_close_hits,
           _test_trigger_spacing_survives_vad_gaps, _test_trigger_hangover_blocks_immediate_retrigger,
           _test_trigger_stale_confirmations_expire]:
    fn()
print("All WakeWordTrigger sanity tests passed.")


## 12. Streaming inference (sliding window)

Includes a cheap energy-based VAD gate so the encoder only runs on frames
that plausibly contain speech -- real AEC (WebRTC AEC3) and a production VAD
(Silero VAD) belong in your live audio front-end, upstream of this function;
this notebook covers detection, not the full deployed pipeline. This gate is
a fixed loudness threshold: loud non-speech (TV, music) will pass it and
reach the encoder just like real speech would -- a known contributor to
false accepts until it's replaced with real VAD.

**Fixed an off-by-one:** the sliding-window loop's stop bound was
`wav.shape[0] - win`, which is *exclusive* in `range()` -- so the one valid
window starting exactly at `wav.shape[0] - win` (the final complete window
in the clip) was silently never scored. Fixed to `wav.shape[0] - win + 1`.
Not a false-accept risk, but it could miss a wake word spoken right at the
end of a short recording.

Confirmation timing is now handled inside `WakeWordTrigger` itself via real
timestamps (Section 11) -- this loop just needs to pass the correct `t` on
every VAD-accepted frame; no separate stride/call-count bookkeeping needed
here anymore.

In [ ]:
VAD_THRESHOLD_DB = -40

def simple_energy_vad(wav_segment, threshold_db=VAD_THRESHOLD_DB):
    rms = wav_segment.pow(2).mean().sqrt()
    db = 20 * torch.log10(rms.clamp_min(1e-8))
    return db.item() > threshold_db

def detect_wakeword(path, hop_sec=SCORING_HOP_SEC):
    wav = load_wav(path)
    hop = int(hop_sec * SR)
    win = CLIP_LEN
    trigger_logic = WakeWordTrigger(profile["threshold"])
    triggers = []
    # FIXED: +1 so the final complete window (start == len - win) is
    # included -- range()'s stop bound is exclusive, so omitting +1 silently
    # dropped the last valid window every time.
    last_start = max(1, wav.shape[0] - win + 1)
    for start in range(0, last_start, hop):
        seg = wav[start:start + win]
        if seg.shape[0] < win:
            seg = F.pad(seg, (0, win - seg.shape[0]))
        t = start / SR
        if not simple_energy_vad(seg):
            continue
        z = embed_wav(encoder, seg)
        score = mahalanobis_score(z, profile)
        if trigger_logic.update(score, t):
            triggers.append(round(t, 2))
    return triggers

# Example:
# print(detect_wakeword("/path/to/some_test_recording.wav"))

## 12.5 Interactive detection demo (upload audio, see the score curve)

This is the piece the earlier simple classifier notebook had and this one was
missing: an actual end-to-end run against real audio, with a plot, not just a
threshold number. Upload a longer test recording (ideally 10-30s) containing
your wake word somewhere in it -- in Colab this prompts a file picker; outside
Colab, set `TEST_AUDIO_PATH` manually below.

The plotted "confidence" is a monotonic 0-1 display transform of the real
Mahalanobis score (`threshold / (threshold + score)`, so 0.5 lines up exactly
with the profile's actual decision boundary) -- it's for readability only.
The trigger markers come from running the real `WakeWordTrigger` state
machine over the stream, the same temporal-confirmation/debounce/hangover
logic `detect_wakeword` uses, not a naive per-window threshold crossing.

In [ ]:
try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Upload a test audio file (ideally 10-30s, containing the wake word somewhere in it).")
    _uploaded = _colab_files.upload()
    TEST_AUDIO_PATH = list(_uploaded.keys())[0]
else:
    TEST_AUDIO_PATH = "/path/to/your_test_recording.wav"  # <-- set this if not running in Colab
    print(f"Not running in Colab -- set TEST_AUDIO_PATH manually (currently: {TEST_AUDIO_PATH})")

import matplotlib.pyplot as plt

def score_to_confidence(score, threshold):
    # Monotonic transform of the Mahalanobis distance into an intuitive 0-1
    # value for plotting only -- 0.5 corresponds exactly to score==threshold.
    # The actual trigger decision below still runs on the real score/threshold
    # via WakeWordTrigger, completely unaffected by this display transform.
    return float(threshold / (threshold + max(score, 0.0)))

def run_detection_demo(path, hop_sec=SCORING_HOP_SEC):
    wav = load_wav(path)
    hop = int(hop_sec * SR)
    win = CLIP_LEN
    trig = WakeWordTrigger(profile["threshold"])
    last_start = max(1, wav.shape[0] - win + 1)

    times, scores, confidences, vad_passed = [], [], [], []
    trigger_events = []

    for start in range(0, last_start, hop):
        seg = wav[start:start + win]
        if seg.shape[0] < win:
            seg = F.pad(seg, (0, win - seg.shape[0]))
        t = start / SR
        passed = simple_energy_vad(seg)
        times.append(t)
        vad_passed.append(passed)
        if not passed:
            scores.append(float("nan"))
            confidences.append(float("nan"))
            continue
        z = embed_wav(encoder, seg)
        score = mahalanobis_score(z, profile)
        scores.append(score)
        confidences.append(score_to_confidence(score, profile["threshold"]))
        if trig.update(score, t):
            trigger_events.append(t)

    return wav, np.array(times), np.array(scores), np.array(confidences), np.array(vad_passed), trigger_events

wav, times, scores, confidences, vad_passed, trigger_events = run_detection_demo(TEST_AUDIO_PATH)

print(f"Scanned {times[-1] + CLIP_SEC:.1f}s of audio using threshold_source='{profile.get('threshold_source', 'unknown')}' "
      f"(threshold={profile['threshold']:.2f}). {len(trigger_events)} confirmed wake-word trigger(s):")
if trigger_events:
    for t in trigger_events:
        idx = int(np.argmin(np.abs(times - t)))
        print(f"  >> WAKE WORD around t={t:.2f}s (confidence {confidences[idx]:.2f})")
else:
    print("  No confirmed trigger. If you expected one, check in order: (1) Section 9's printed")
    print("  threshold_source -- if it says 'target-FAR calibration' rather than 'relaxed-for-")
    print("  usability', the strict threshold may still be too tight for this profile; (2) does")
    print("  this recording's speaker/conditions actually match the enrolled profile; (3) is")
    print("  VAD_THRESHOLD_DB screening out quiet audio before it ever reaches the encoder.")

fig2, ax = plt.subplots(figsize=(14, 4))
ax.plot(times, confidences, color="steelblue", label="confidence (0-1, higher = more wake-word-like)")
ax.axhline(0.5, color="red", linestyle="--", label="trigger boundary (score == profile threshold)")
for t in trigger_events:
    ax.axvline(t, color="green", alpha=0.6)
    idx = int(np.argmin(np.abs(times - t)))
    ax.scatter([t], [confidences[idx]], color="green", zorder=5)
not_vad = ~vad_passed
ax.fill_between(times, 0, 1, where=not_vad, color="grey", alpha=0.12, step="mid",
                 label="VAD-gated (skipped)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Confidence")
ax.set_ylim(-0.02, 1.02)
ax.set_title(f"Wake word detection demo — {len(trigger_events)} confirmed trigger(s)")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("detection_demo.png", dpi=130)
plt.show()

writer.add_scalar("detection_demo/num_triggers", len(trigger_events), 0)
writer.add_figure("detection_demo/confidence_over_time", fig2, 0)
writer.flush()

try:
    from IPython.display import Audio, display
    print("\\nPlayback of tested audio:")
    display(Audio(wav.numpy(), rate=SR))
except Exception:
    pass

### Continuous-listening false activation estimate

The detector's operational metric is false activations per hour, not only per-window FAR.
This helper runs the full trigger logic over non-wake audio and measures the rate using real
stream timestamps. It is an evaluation metric only and does not change training.

In [ ]:
def false_activations_per_hour(encoder, profile, audio_wavs, max_hours=2.0):
    """Run the COMPLETE trigger logic over non-wake audio and report false activations/hour.

    audio_wavs should contain held-out non-wake recordings (speech/noise). The scoring
    window and temporal confirmation logic are exactly the same as detect_wakeword().
    This is intentionally not used for calibration.
    """
    total_seconds = 0.0
    false_triggers = 0
    for wav in audio_wavs:
        if total_seconds >= max_hours * 3600:
            break
        duration = wav.shape[0] / SR
        remaining = max_hours * 3600 - total_seconds
        if duration > remaining:
            wav = wav[:int(remaining * SR)]
            duration = wav.shape[0] / SR
        if duration < CLIP_LEN:
            continue
        trigger_logic = WakeWordTrigger(profile["threshold"])
        hop = int(SCORING_HOP_SEC * SR)
        last_start = wav.shape[0] - CLIP_LEN + 1
        for start in range(0, last_start, hop):
            seg = wav[start:start + CLIP_LEN]
            t = (total_seconds + start / SR)
            if not simple_energy_vad(seg):
                continue
            score = mahalanobis_score(embed_wav(encoder, seg), profile)
            false_triggers += int(trigger_logic.update(score, t))
        total_seconds += duration

    if total_seconds <= 0:
        return {"false_triggers": 0, "hours": 0.0, "false_activations_per_hour": float("nan")}
    return {
        "false_triggers": int(false_triggers),
        "hours": total_seconds / 3600.0,
        "false_activations_per_hour": float(false_triggers / (total_seconds / 3600.0)),
    }


## 13. Export encoder to ONNX (deployment)

Export once per backbone (Layer 1), not per wake word -- the enrollment
profile (`wakeword_profile.json`) is a tiny separate artifact that ships
alongside it and is what actually changes per wake word/user.

In [ ]:
dummy_time = 101   # matches CLIP_LEN=16000 samples at hop_length=160 -> ~101 frames
dummy = torch.randn(1, 1, N_MELS, dummy_time).to(device)

torch.onnx.export(
    encoder, dummy, "encoder.onnx",
    input_names=["log_mel_pcen"], output_names=["embedding"],
    dynamic_axes={"log_mel_pcen": {0: "batch", 3: "time"}, "embedding": {0: "batch"}},
    opset_version=17,
)
print("Exported encoder.onnx -- quantize with onnxruntime.quantization for the final deployed size.")

### 13.1 Verify the ONNX export actually matches the PyTorch model

Exporting silently "succeeding" is not the same as exporting *correctly*. Dynamic control
flow, the GRU temporal head, and PCEN's frame-by-frame recurrence are exactly the kind of
thing that can export without error but produce numerically different output (a wrong
opset behavior, a silently-frozen dynamic axis, a GRU direction/layout mismatch, etc.).
This runs both the frozen PyTorch encoder and the exported ONNX graph on the same real
enrollment/calibration clips and checks they agree -- catching an export bug here, before
it reaches a device, is a lot cheaper than debugging degraded accuracy in the field.

In [ ]:
import onnxruntime as ort

def _torch_embed_batch(encoder, wavs):
    feats = torch.cat([wav_to_features(fix_length_center(w)) for w in wavs], dim=0)
    with torch.no_grad():
        return encoder(feats.to(device)).cpu().numpy()

def _onnx_embed_batch(session, wavs):
    feats = torch.cat([wav_to_features(fix_length_center(w)) for w in wavs], dim=0).numpy().astype(np.float32)
    (out,) = session.run(["embedding"], {"log_mel_pcen": feats})
    return out

def check_onnx_parity(onnx_path, files, max_files=16, atol=1e-4, rtol=1e-3):
    files = files[:max_files]
    assert files, "No files available to run the ONNX parity check against."
    wavs = [load_wav(f) for f in files]

    session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    torch_out = _torch_embed_batch(encoder, wavs)
    onnx_out = _onnx_embed_batch(session, wavs)

    max_abs_diff = float(np.max(np.abs(torch_out - onnx_out)))
    cos_sim = float(np.mean(np.sum(torch_out * onnx_out, axis=1) /
                             (np.linalg.norm(torch_out, axis=1) * np.linalg.norm(onnx_out, axis=1) + 1e-9)))
    close = np.allclose(torch_out, onnx_out, atol=atol, rtol=rtol)

    print(f"ONNX parity check over {len(files)} clips:")
    print(f"  max |torch - onnx| embedding diff: {max_abs_diff:.6f}")
    print(f"  mean cosine similarity:            {cos_sim:.6f}")
    print(f"  np.allclose(atol={atol}, rtol={rtol}): {close}")
    if not close:
        print("  WARNING: PyTorch and ONNX outputs diverge beyond tolerance -- do NOT deploy this")
        print("  export as-is. Common causes: opset version, GRU export quirks, PCEN's frame-by-frame")
        print("  recurrence not lowering cleanly. Try a different opset_version or re-check the graph.")
    return {"max_abs_diff": max_abs_diff, "cosine_sim": cos_sim, "allclose": close}

_parity_probe_files = (enrollment_files or [])[:8] + (positive_eval_files or [])[:8]
if not _parity_probe_files:
    _parity_probe_files = test_other_files[:16]
_ = check_onnx_parity("encoder.onnx", _parity_probe_files)


### 13.2 Quantize for deployment (and check what it costs you)

Section 13's original note said "quantize with `onnxruntime.quantization` for the final
deployed size" but never actually ran it. Quantizing without measuring the accuracy cost is
how a keyword spotter quietly gets worse on-device than it tested in this notebook. This
cell performs real INT8 dynamic quantization, reports the size reduction, and reruns the
parity check (against the *floating-point* PyTorch model, so the reported diff is the true
end-to-end cost of quantizing) so you can see the actual trade-off before shipping it.

In [ ]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

QUANTIZED_ONNX_PATH = "encoder.int8.onnx"

quantize_dynamic(
    model_input="encoder.onnx",
    model_output=QUANTIZED_ONNX_PATH,
    weight_type=QuantType.QInt8,
)

fp32_size = os.path.getsize("encoder.onnx")
int8_size = os.path.getsize(QUANTIZED_ONNX_PATH)
print(f"encoder.onnx:      {fp32_size / 1024:8.1f} KB")
print(f"encoder.int8.onnx: {int8_size / 1024:8.1f} KB  ({100 * int8_size / fp32_size:.1f}% of fp32 size)")

print("\nRe-running the parity check against the QUANTIZED model (compared to the original")
print("floating-point PyTorch encoder, not the fp32 ONNX export) to measure the real cost:")
quant_parity = check_onnx_parity(QUANTIZED_ONNX_PATH, _parity_probe_files, atol=2e-2, rtol=5e-2)
if quant_parity["cosine_sim"] < 0.98:
    print("\nWARNING: quantized embeddings diverge noticeably from the fp32 model. Since the")
    print("enrollment profile (mean/covariance) was fit on FP32 embeddings, re-run enrollment")
    print("(Section 8) with the quantized model's embeddings if you deploy this quantized graph,")
    print("rather than mixing an fp32-fitted profile with int8 inference embeddings.")
else:
    print("\nQuantized embeddings stay close enough to fp32 that the existing fp32-fitted profile")
    print("should still be usable, but re-validate FAR/FRR (Section 10) end-to-end with the")
    print("quantized model before trusting that in production.")


## 13.5 Standalone deployment inference (no PyTorch, no training globals)

Everything above -- including `detect_wakeword` in Section 12 -- quietly depends on
in-notebook state (`encoder` as a live `nn.Module`, `profile` as a Python dict already in
memory). That's fine for experimentation, but it's not what actually runs on a device: a
real deployment only has `encoder.onnx` (or `encoder.int8.onnx`) and `wakeword_profile.json`
as files, loaded fresh, with `onnxruntime` and `numpy` only -- no `torch`. This section
re-implements the streaming detector against exactly those two artifacts, as a sanity check
that they're actually sufficient on their own before you wire them into a device build.

In [ ]:
"""Standalone reference implementation -- copy this cell (plus the feature-extraction
helpers it needs) into your actual deployment codebase. It intentionally avoids every
notebook/training global except pure functions and numpy/onnxruntime, to prove the two
shipped artifacts (encoder.onnx + wakeword_profile.json) are self-contained."""

import json as _json
import numpy as _np
import onnxruntime as _ort


class StandaloneWakeWordDetector:
    def __init__(self, onnx_path, profile_path, sr=SR, clip_sec=CLIP_SEC,
                 vad_threshold_db=VAD_THRESHOLD_DB, hop_sec=SCORING_HOP_SEC):
        self.session = _ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        with open(profile_path) as f:
            self.profile = _json.load(f)
        self.sr = sr
        self.clip_len = int(sr * clip_sec)
        self.hop = int(hop_sec * sr)
        self.vad_threshold_db = vad_threshold_db
        self.trigger = WakeWordTrigger(self.profile["threshold"])

        self.mean = _np.asarray(self.profile["mean"], dtype=_np.float64)
        self.cov_inv = _np.asarray(self.profile["cov_inv"], dtype=_np.float64)
        self.cov_mode = self.profile["cov_mode"]
        self.pca_basis = self.profile.get("pca_basis")

    def _vad(self, seg):
        rms = float(_np.sqrt(_np.mean(seg.astype(_np.float64) ** 2)))
        db = 20 * _np.log10(max(rms, 1e-8))
        return db > self.vad_threshold_db

    def _embed(self, wav_np):
        wav_t = torch.from_numpy(wav_np).float()
        feat = wav_to_features(fix_length_center(wav_t)).numpy().astype(_np.float32)
        (out,) = self.session.run(["embedding"], {"log_mel_pcen": feat})
        return out.squeeze(0).astype(_np.float64)

    def _score(self, z):
        if self.cov_mode in ("pca_diag", "pca") and self.pca_basis is not None:
            comp = _np.asarray(self.pca_basis["components"], dtype=_np.float64)
            zp = z @ comp.T
        else:
            zp = z
        diff = zp - self.mean
        return float(diff @ self.cov_inv @ diff.T)

    def process_stream(self, wav_np):
        """wav_np: float32 mono numpy array at self.sr. Returns trigger timestamps (sec)."""
        triggers = []
        last_start = max(1, wav_np.shape[0] - self.clip_len + 1)
        for start in range(0, last_start, self.hop):
            seg = wav_np[start:start + self.clip_len]
            if seg.shape[0] < self.clip_len:
                seg = _np.pad(seg, (0, self.clip_len - seg.shape[0]))
            t = start / self.sr
            if not self._vad(seg):
                continue
            score = self._score(self._embed(seg))
            if self.trigger.update(score, t):
                triggers.append(round(t, 2))
        return triggers


# Smoke test: the fp32 ONNX path should reproduce the in-notebook detect_wakeword() output
# on the same audio, since both score against the same profile with the same trigger logic.
if enrollment_files:
    _probe_wav, _ = torchaudio.load(enrollment_files[0])
    _probe_wav = load_wav(enrollment_files[0]).numpy().astype(np.float32)
    _detector = StandaloneWakeWordDetector("encoder.onnx", PROFILE_PATH)
    print("Standalone (onnxruntime-only) detector triggers on enrollment clip 0:",
          _detector.process_stream(_probe_wav))
    print("This should behave like detect_wakeword() on the same clip -- both are scoring")
    print("against the identical profile with the identical trigger logic, just via different")
    print("model runtimes (torch vs onnxruntime).")


## 14. Interpretation of results

`TARGET_FAR` is a calibration target, not a guarantee. The threshold is calibrated only
from the DISJOINT public calibration pool plus the disjoint held-out noise-calibration chunk.
If the calibration pool is too small for a direct empirical 1e-4 percentile, the notebook
uses a Gamma-tail extrapolation. Section 10's completely untouched official test set is
the source of truth for achieved FAR/FRR.

For a meaningful genuine-side result, put 20–100 genuinely new wake-word recordings
into `POSITIVE_EVAL_DIR`, with variation in room, distance, microphone, speaking rate,
background and day/session. Three to five enrollment clips are enough to create a profile,
but they do not prove universal generalization.

For continuous-listening performance, also report **false activations per hour** on long
non-wake audio. A low per-window FAR is not equivalent to a low device-level false-trigger
rate because the detector scores many windows per hour and uses overlapping context.


## Re-enrolling for a new wake word, or fixing a struggling user

Delete/replace the clips in `ENROLLMENT_DIR`, then rerun Sections 8–10. No negative
wake-word data or Layer-2 retraining is required.

### v7 changes (bug fixes only, same design as v6)

- **Fixed:** `train_class_banks` was used in the Section 6 training loop but never
  defined -- this was a guaranteed `NameError` on the first training episode. Now built
  from `train_groups` over `KEYWORD_CLASSES`, mirroring how `val_class_banks` is built.
- **Fixed:** `ENROLLMENT_DIR` and `POSITIVE_EVAL_DIR` are now created automatically
  (`os.makedirs(..., exist_ok=True)`) so a missing folder fails with a clear assert
  message instead of a silent empty `glob()`.

### v9 changes (fixes detection never firing, adds detection demo + TensorBoard)

- **Fixed the actual bug behind "sets a threshold and never detects anything":** Section 9
  computed a `TARGET_FAR`-calibrated threshold that, with only 3-5 real enrollment clips, is
  very often tighter than the enrollment spread itself -- the old code printed a warning about
  this and then saved the too-tight threshold anyway. Since `WakeWordTrigger` fires on
  `score < threshold`, once this happened even the enrolled speaker's OWN genuine wake-word
  utterances scored above threshold, so the detector became mathematically incapable of firing
  for anyone. Section 9 now auto-relaxes to a usable threshold when this happens, logs which
  threshold source was actually used (`profile["threshold_source"]`), and keeps the original
  target-FAR value visible (`profile["target_far_threshold"]`) rather than silently discarding it.
- **Added the missing interactive detection demo (Section 12.5)** -- upload/point at real test
  audio, scan it, plot a confidence curve with actual confirmed-trigger markers (via the real
  `WakeWordTrigger` state machine, not a naive per-window threshold check), and play it back.
  This was the piece the simpler classifier notebook had that made its results visible and
  trustworthy, and this notebook was missing entirely.
- **Added TensorBoard** -- Layer-1 training curves (train/val loss + accuracy per epoch),
  Section 9's calibration threshold values, Section 10's FAR/FRR/EER, the score-distribution
  and DET-curve figure, and the detection-demo figure and trigger count all log to one
  timestamped run directory, viewable inline via the dashboard cell near the top of the
  notebook -- training and testing tracked in the same place, not read off scattered prints.

### v8 changes (added the missing verification/production layer -- no changes to Layer 1/2 math)

- **Added:** environment/version snapshot printed right after imports, since the install
  cell is intentionally unpinned and drift over time was previously undiagnosable from the
  notebook alone.
- **Added:** score-distribution histograms and a DET curve (FAR vs. FRR swept over
  threshold) for Section 10's held-out evaluation -- a single EER number can hide a
  bimodal or heavy-tailed distribution that's worth seeing directly.
- **Added:** automated sanity tests for `WakeWordTrigger` (Section 11.5), including a
  direct regression test for the exact VAD-gap timestamp bug the v7 changelog fixed --
  that class had zero test coverage anywhere in the notebook before this.
- **Added:** a real PyTorch\u2194ONNX numerical parity check after export (Section 13.1).
  The v7 notebook exported to ONNX and declared success on the export call not raising --
  that doesn't mean the graph is numerically correct, and this now actually verifies it on
  real clips before you trust the exported model.
- **Added:** real INT8 dynamic quantization (Section 13.2), replacing a comment that told
  you to quantize but never did it. Reports the actual size reduction and re-runs the
  parity check against the quantized graph so the accuracy trade-off is measured, not
  assumed -- and warns you to re-fit the profile if quantization moves the embeddings too
  far from what the profile was calibrated against.
- **Added:** a standalone `onnxruntime`-only reference detector (Section 13.5) that loads
  only the two artifacts a real device actually ships with (`encoder.onnx` +
  `wakeword_profile.json`) with no PyTorch and no notebook globals -- proving those two
  files are actually sufficient on their own, which nothing in v7 verified.

### v6 changes

- Public Speech Commands validation data is split into three disjoint roles: model selection,
  statistics/PCA learning, and threshold-calibration speech. The untouched official test set
  remains final evaluation only.
- Layer 2 defaults to `pca_diag` with **8 dimensions** and diagonal inverse covariance.
- PCA directions and the public variance prior are learned only from the disjoint statistics split.
- Enrollment statistics remain clip-balanced: each real recording contributes one clip mean,
  while augmentation contributes only a within-clip variance term.
- Threshold calibration speech comes from a separate public calibration split, eliminating the
  previous reuse of the same validation clips for both PCA/statistics and threshold selection.
- Noise calibration and test windows remain non-overlapping in time.
- Temporal confirmation is timestamp-based and the final complete sliding window is included.
- Section 10 continues to warn when the fresh positive evaluation set is too small to support
  a meaningful FRR estimate.

### Production boundary

This notebook is internally consistent for research/prototype use. A real product still needs
a proper live-audio frontend (AEC, noise suppression and a real VAD) and a streaming feature
cache so the same 1-second context is not recomputed from scratch every 100 ms.

The positive-only philosophy is preserved: only genuine wake-word recordings are enrolled.
Public speech/noise is used to build the universal encoder and to calibrate/evaluate the detector,
not as wake-word-specific Layer-2 training data.
